# 8장 — RAG 조립 🏗️ (핵심)

두 파이프라인(인덱싱·질의)을 완전판으로 조립하고, 실패 모드를 코드로 재현하고,
갈림길(조련 vs RAG) 판단을 함수로 만들어 봅니다.

**이 노트북의 목표**
1. 인덱싱(청킹→임베딩→저장)과 질의(검색→그라운딩 프롬프트→생성)를 분리해 조립
2. 실패 모드 재현 — 문서 부재 시 "모른다"로 답하게 만들기(그라운딩)
3. 갈림길 판단 규칙을 코드로 — 지식은 RAG, 행동은 조련, 둘 다면 병행

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch8')

## 1. 실험 ① — 인덱싱 파이프라인 (미리, 오프라인)

문서 → 조항 청킹(7장) → toy 임베딩 → "벡터 DB"(여기선 리스트). 제목 붙이기 요령도 적용합니다.

In [ ]:
import numpy as np

def normalize(v):
    return v / np.linalg.norm(v)

# toy 임베딩: 주제 축 6개 (결혼, 장례, 카드정산, 연차, 절차, 행사) — 실제는 모델이 계산(9장)
TOPIC = {'결혼': 0, '장례': 1, '카드': 2, '연차': 3, '절차': 4, '행사': 5}
def embed(topic_weights):
    v = np.zeros(6)
    for t, w in topic_weights.items():
        v[TOPIC[t]] = w
    return normalize(v)

# 문서 2건 → 조항 청킹 + 제목 붙이기 (인덱싱은 느려도 되는 배치 작업)
DB = [
    {'title': '경조사 규정 > 제1조(결혼)', 'text': '결혼 축하금 50만 원. 단, 근속 1년 이상.', 'vec': embed({'결혼': 1.0, '절차': 0.1})},
    {'title': '경조사 규정 > 제2조(장례)', 'text': '장례 지원금 100만 원과 상조 서비스.', 'vec': embed({'장례': 1.0})},
    {'title': '경비 규정 > 제3조(정산)', 'text': '법인카드는 매월 5일까지 경비 시스템에 등록.', 'vec': embed({'카드': 1.0, '절차': 0.2})},
    {'title': '인사 규정 > 제7조(연차)', 'text': '연차는 인사 포털에서 결재선을 지정해 신청.', 'vec': embed({'연차': 1.0, '절차': 0.3})},
]
print(f'인덱싱 완료: 조각 {len(DB)}개 (청킹 + 제목 붙이기 + 임베딩 + 저장)')

## 2. 실험 ② — 질의 파이프라인 + 그라운딩

검색(top-k) → **그라운딩 프롬프트 3문장** → 생성(흉내). 점수가 바닥이면 "모른다"고 답하게 합니다.

In [ ]:
GROUNDING = ('아래 사내 문서에 근거해서만 답하세요.\n'
             '문서에 없는 내용은 "문서에서 찾지 못했습니다"라고 답하세요.\n'
             '답변 끝에 참고한 문서 제목을 표기하세요.\n')

def rag_answer(q_text, q_vec, k=2, floor=0.5):
    scores = sorted(((float(d['vec'] @ q_vec), d) for d in DB), key=lambda s: -s[0])
    top = [(s, d) for s, d in scores[:k]]
    print('📋 검색 로그: ' + ', '.join(f"{d['title'].split('>')[1].strip()} ({s:.2f})" for s, d in top))
    if top[0][0] < floor:                       # 재료가 바닥 수준 → 정직한 모름 (그라운딩 둘째 줄)
        return '문서에서 찾지 못했습니다. 담당 부서에 문의해 주세요.'
    prompt = GROUNDING + '\n'.join(f"[문서] {d['title']} — {d['text']}" for _, d in top) + f'\n[질문] {q_text}'
    best = top[0][1]                            # 생성 흉내: 1등 조각으로 답 (진짜 LLM은 10장)
    return f"{best['text']} (출처: {best['title']})"

# 정상 질의 — 결혼 질문
ans1 = rag_answer('결혼 축하금 얼마예요?', embed({'결혼': 0.95, '절차': 0.2}))
print('🤖', ans1, '\n')
assert '50만 원' in ans1 and '근속 1년' in ans1, '조항 청킹 덕에 조건까지 온전한 답'
assert '출처' in ans1, '그라운딩 셋째 줄 — 출처 표기'

# 실패 모드 4(문서 부재) — 워크숍 질문: DB에 없는 주제 → 지어내지 않고 정직하게 모른다
ans2 = rag_answer('겨울 워크숍 언제예요?', embed({'행사': 1.0, '절차': 0.15}))
print('🤖', ans2)
assert '찾지 못했습니다' in ans2, '재료가 바닥이면 환각 대신 정직한 모름 — 그라운딩'
print('\n✔ 확인: 같은 파이프라인이 "아는 것은 출처와 함께, 모르는 것은 모른다고" 답한다')

## 3. 실험 ③ — 갈림길을 함수로: 조련 vs RAG vs 병행

실무 결정 질문 두 개 — "바뀌는 지식/출처가 필요한가?", "말투·형식·행동인가?" — 를 그대로 코드로.

In [ ]:
def choose_path(needs_knowledge, needs_behavior):
    """지식(사실·갱신·출처)이면 RAG, 행동(말투·형식·경계)이면 조련, 둘 다면 병행."""
    if needs_knowledge and needs_behavior:
        return '병행 (지식은 RAG로 건네고, 행동은 조련으로 새긴다)'
    if needs_knowledge:
        return 'RAG'
    if needs_behavior:
        return '조련 (파인튜닝·정렬)'
    return '둘 다 불필요 (프롬프트만으로 충분할 수도)'

cases = [
    ('매주 바뀌는 출장비 규정을 답하게', True, False, 'RAG'),
    ('답변을 회사 공문 말투로', False, True, '조련'),
    ('최신 규정을 공문 말투로 + 출처 필수', True, True, '병행'),
    ('보안 위반 요청은 정중히 거절', False, True, '조련'),
]
for desc, k_, b_, expect in cases:
    path = choose_path(k_, b_)
    print(f'  {desc:32s} → {path}')
    assert path.startswith(expect), f'{desc} 는 {expect} 이어야'

print('\n✔ 확인: 지식은 RAG, 행동은 조련, 둘 다면 병행 — 상용 AI 비서의 실제 조합(6부+7부)')

## 4. 실험 ④ — 벡터로 부족할 때: 관계 지도(지식 그래프) 두 걸음 걷기

"지원금 **결재자의 상급 부서**는?" — 벡터 검색은 '경조사 지원' 조각 하나만 준다. 답은 그 조각에 없고
**관계**(결재자 → 소속)를 따라 두 걸음 가야 나온다(다단계 질문, multi-hop).
관계를 (출발, 관계, 도착) 3중항으로 적어 두고(= **지식 그래프**, knowledge graph — 이 과정의 비유어는 관계 지도)
벡터 검색이 준 출발점에서 화살표를 따라 이웃을 함께 싣는다 — **GraphRAG** 의 최소 형태.

In [ ]:
# 관계 지도 = 지식 그래프: (출발, 관계, 도착) 3중항. 실무에선 직제·조직도처럼 이미 표가 있는 관계부터.
edges = [
    ('경조사 지원 규정', '결재자', '총무팀장'),
    ('총무팀장', '소속', '경영지원본부'),
    ('출장비 규정', '결재자', '재무팀장'),
    ('재무팀장', '소속', '경영지원본부'),
    ('출장비 규정', '참조', '법인카드 규정'),
]

def neighbors(node, relation=None):
    """node 에서 나가는 화살표를 따라 한 걸음"""
    return [(r, dst) for src, r, dst in edges if src == node and (relation is None or r == relation)]

def walk(start, path):
    """관계 이름의 순서(path)대로 걷는다 — 다단계 질문(multi-hop)의 최소 구현"""
    node, trail = start, [start]
    for rel in path:
        nxt = neighbors(node, rel)
        if not nxt:
            return trail, None
        node = nxt[0][1]
        trail.append(node)
    return trail, node

# ① 벡터 검색이 '경조사 지원 규정' 조각을 1등으로 줬다고 하자 — 여기가 출발점
start = '경조사 지원 규정'
trail, answer = walk(start, ['결재자', '소속'])
print(' → '.join(trail))
assert answer == '경영지원본부'

# ② 거꾸로 걷기 — "법인카드 규정을 바꾸면 영향받는 문서는?" : '참조' 화살표를 반대로
affected = [src for src, r, dst in edges if r == '참조' and dst == '법인카드 규정']
print('법인카드 규정을 참조하는 문서:', affected)
assert affected == ['출장비 규정']

# ③ GraphRAG 최소형: 출발 조각 + 두 걸음 안의 이웃을 함께 프롬프트에 싣는다
context = [start] + [dst for _, dst in neighbors(start)] + [dst for _, dst in neighbors('총무팀장')]
print('프롬프트에 실을 조각:', context)
assert context == ['경조사 지원 규정', '총무팀장', '경영지원본부']
print('\n✔ 확인: 벡터 검색은 출발점을, 관계 지도는 그다음 걸음을 준다 — 다단계 질문은 둘을 합쳐야 답이 나온다')

## 5. 관찰 정리 + 🤖 AI 연결

- **두 파이프라인**: 인덱싱(미리 — 청킹·임베딩·저장)과 질의(실시간 — 검색·조립·생성)를 분리.
  고장의 절반은 인덱싱에서 나는데 증상은 질의에서 보인다 — 진단은 **검색 로그부터**.
- **4대 실패 모드**: 검색 실패 · 조각 찢김 · 생성 실패(문맥 무시) · 문서 부재.
  그라운딩 3문장(근거해서만 / 없으면 없다고 / 출처 표기)이 뒤의 둘을 막는다.
- **갈림길**: 지식은 RAG(6부 2장 — 문맥 창), 행동은 조련(6부 3장 — SFT: Supervised
  Fine-Tuning, 지도 파인튜닝), 둘 다면 병행.

- **벡터로 부족할 때**: 관계(결재자·소속·참조)를 따라 걷는 관계 지도(지식 그래프, knowledge graph) —
  다단계 질문은 벡터 검색(출발점) + 두 걸음(GraphRAG). 이 시리즈의 튜터·개념 띠·개념 지도가 같은 방식(concept-graph.json).

**다음 챕터**: 9장 — 임베딩은 어떻게 배우나. 이 모든 것의 토대인 좌표를 모델이 어떻게 배우는지 —
"**짝은 가깝게, 남은 멀게**"(대조 학습), 그리고 또 만나는 −log 손실(5부).